# SSL400 ST-GCN — Google Colab Training (accuracy + mild augmentation)

The all-class run (~383 classes) dropped accuracy to about **25%** because most rare signs have only 1–2 samples, and strong balancing/augmentation fought overall accuracy.

This notebook trains classes with **≥ 10 samples** (the setup that reached ~54%) and adds **mild** landmark augmentation only.

## Before you start

1. Upload to Drive:
```text
MyDrive/SSL400/
  ssl400-research/
  Dataset - MP - CSV/
```
2. **Runtime → Change runtime type → GPU (T4)**
3. Run cells **in order**.
4. This run **rebuilds** `data/processed/` (`FORCE_REBUILD = True`) because the class filter changed.

## 1) Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2) Go to project folder

Edit paths if needed.

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/SSL400/ssl400-research')
DATASET_DIR = Path('/content/drive/MyDrive/SSL400/Dataset - MP - CSV')

assert PROJECT_DIR.exists(), f'Missing project: {PROJECT_DIR}'
assert DATASET_DIR.exists(), f'Missing dataset: {DATASET_DIR}'

os.chdir(PROJECT_DIR)
print('CWD:', Path.cwd())
print('Project:', sorted(p.name for p in PROJECT_DIR.iterdir())[:20])

## 3) Check GPU

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — switch runtime to T4')

## 4) Install dependencies

In [ ]:
!pip install -q -r requirements.txt
print('Install done')

## 5) Write configs (classes ≥ 10 + mild augmentation)

- `min_samples_per_class: 10`
- Mild augmentation only (small scale, noise, rotation, time shift)
- No weighted sampler and no class weights (those lowered overall accuracy)

In [ ]:
from pathlib import Path

Path('configs').mkdir(exist_ok=True)

Path('configs/data.yaml').write_text(f'''
raw_csv_root: "{DATASET_DIR.as_posix()}"
processed_dir: "data/processed"
inspected_dir: "data/inspected"
min_samples_per_class: 10
sequence_length: 64
use_z: true
visibility_threshold: 0.5
mask_low_visibility: true
mask_lower_body: false
train_ratio: 0.70
val_ratio: 0.15
test_ratio: 0.15
split_seed: 42
save_npz: true
'''.strip() + '\n', encoding='utf-8')

Path('configs/model.yaml').write_text('''
model:
  in_channels: 3
  num_joints: 33
  num_classes: null
  channels: [64, 64, 128, 256]
  temporal_kernel: 9
  dropout: 0.5
  graph_strategy: spatial

train:
  epochs: 100
  batch_size: 32
  learning_rate: 0.001
  weight_decay: 0.0001
  optimizer: adam
  scheduler: cosine
  step_size: 30
  gamma: 0.1
  early_stopping_patience: 20
  num_workers: 0
  amp: true
  class_weights: false
  weighted_sampler: false
  label_smoothing: 0.0
  focal_gamma: 1.0
  augment: true
  seed: 42
  device: auto

paths:
  checkpoint_dir: "checkpoints"
  results_dir: "results"
  plots_dir: "results/plots"
  experiment_name: "stgcn_lowres_seed42"
'''.strip() + '\n', encoding='utf-8')

print('Wrote configs (min_samples_per_class=10, mild augmentation)')

## 6) Prepare dataset (rebuild for classes with ≥ 10 samples)

Keep `FORCE_REBUILD = True` for this new run so the old all-class tensors are not reused.

In [ ]:
import shutil
from pathlib import Path

FORCE_REBUILD = True  # set False after first successful prepare

processed = Path('data/processed')
if FORCE_REBUILD and processed.exists():
    shutil.rmtree(processed)
    print('Deleted old data/processed for full rebuild')

!python scripts/prepare_dataset.py --data-config configs/data.yaml

In [ ]:
import json
from pathlib import Path
import numpy as np

class_map = json.loads(Path('data/processed/class_to_idx.json').read_text(encoding='utf-8'))
manifest = json.loads(Path('data/processed/manifest.json').read_text(encoding='utf-8'))
sample = manifest['train'][0]
tensor = np.load(sample['tensor_path'])

print('Classes (should be about 121, not 383):', len(class_map))
print('Split sizes:', {k: len(v) for k, v in manifest.items()})
print('Example tensor:', sample['sample_id'], tensor.shape, tensor.dtype)

## 7) Train ST-GCN

If CUDA OOM: set `batch_size: 16` in cell 5 and re-run cell 5 + this cell.

In [ ]:
!python scripts/train_stgcn.py --data-config configs/data.yaml --model-config configs/model.yaml

## 8) Evaluate on test set (+ confusion / F1 plots)

In [ ]:
!python scripts/evaluate_stgcn.py \
  --checkpoint checkpoints/stgcn_lowres_seed42_best.pt \
  --split test \
  --data-config configs/data.yaml \
  --model-config configs/model.yaml

## 9) Monte Carlo Dropout (low-resource uncertainty)

Same trained model, dropout left on for 20 passes. Use the **validation** split here. The test split stays for the locked accuracy number in section 8.

High entropy means the model is unsure — that is the signal the research guide later uses for human Top-5 verification.

In [ ]:
!python scripts/mc_dropout_eval.py \
  --checkpoint checkpoints/stgcn_lowres_seed42_best.pt \
  --split val \
  --passes 20 \
  --data-config configs/data.yaml \
  --model-config configs/model.yaml

## 10) Export `.h5`

In [ ]:
from pathlib import Path
import h5py

ckpt = Path('checkpoints/stgcn_lowres_seed42_best.pt')
h5 = Path('checkpoints/stgcn_lowres_seed42_best.h5')
assert ckpt.exists(), ckpt

!python scripts/export_h5.py --checkpoint checkpoints/stgcn_lowres_seed42_best.pt --output checkpoints/stgcn_lowres_seed42_best.h5

with h5py.File(h5, 'r') as f:
    print('H5 OK:', h5, 'classes=', int(f['metadata'].attrs['num_classes']))

## 11) Show all training graphs

Plots are also saved under `results/plots/` on Drive.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

plots_dir = Path('results/plots')
exp = 'stgcn_lowres_seed42'
wanted = [
    f'{exp}_split_sizes.png',
    f'{exp}_class_distribution.png',
    f'{exp}_loss_curves.png',
    f'{exp}_metric_curves.png',
    f'{exp}_overfit_gap.png',
    f'{exp}_lr_curve.png',
    f'{exp}_confusion_top30.png',
    f'{exp}_per_class_f1.png',
    f'{exp}_test_confusion_top30.png',
    f'{exp}_test_per_class_f1.png',
    f'{exp}_val_mc_uncertainty.png',
]

found = []
for name in wanted:
    path = plots_dir / name
    if path.exists():
        found.append(path)
        print('Showing', path.name)
        display(Image(filename=str(path)))
    else:
        print('Missing (may appear after train/eval):', name)

print('\nAll PNGs in plots_dir:')
for p in sorted(plots_dir.glob('*.png')):
    print('-', p.name)

## 12) Summary / download

In [ ]:
import json
from pathlib import Path
from google.colab import files

exp = 'stgcn_lowres_seed42'
summary_path = Path(f'results/{exp}_train_summary.json')
metrics_path = Path(f'results/{exp}_test_metrics.json')
ckpt_path = Path(f'checkpoints/{exp}_best.pt')
h5_path = Path(f'checkpoints/{exp}_best.h5')

if summary_path.exists():
    print('Train summary:')
    print(json.dumps(json.loads(summary_path.read_text()), indent=2))

if metrics_path.exists():
    m = json.loads(metrics_path.read_text())
    keep = {k: m[k] for k in m if k not in ('y_true', 'y_pred', 'sample_ids')}
    print('\nTest metrics:')
    print(json.dumps(keep, indent=2))

print('\n.pt:', ckpt_path.exists(), ckpt_path)
print('.h5:', h5_path.exists(), h5_path)

# Uncomment to download:
# if ckpt_path.exists(): files.download(str(ckpt_path))
# if h5_path.exists(): files.download(str(h5_path))

## Notes

- Low-resource techniques in this run: classes with ≥ 10 samples, mild pose augmentation, signing-speed resample, focal loss (gamma 1), MC Dropout uncertainty.
- Still later (see the development guide): temperature scaling, accept/verify Top-5 UI, verified-only replay, continual learning.
- Graphs: split sizes, class distribution, loss, top-1/top-5/F1, overfit gap, learning rate, confusion, per-class F1, MC uncertainty.
- Pose-only SSL still has a natural accuracy ceiling (no hand landmarks).
- If disconnected: remount Drive, set `FORCE_REBUILD=False` only after prepare finished, then re-run train.